 # 스타벅스 매장 정보 수집 실습

 ## 1. 환경 설정 및 드라이버 함수 정의

In [6]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

def get_chrome_driver():
    """봇 감지를 회피하는 최적화된 Chrome 드라이버 생성"""
    
    chrome_options = Options()
    
    # ========== 기본 옵션 ==========
    chrome_options.add_argument('--start-maximized')
    chrome_options.add_argument('--disable-blink-features=AutomationControlled')
    
    # ========== 자동화 감지 회피 ==========
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option('useAutomationExtension', False)
    
    # ========== User-Agent 설정 ==========
    user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    chrome_options.add_argument(f'user-agent={user_agent}')

    # ========== 드라이버 생성 ==========
    driver = webdriver.Chrome(options=chrome_options)
    
    # ========== JavaScript를 통한 추가 위장 ==========
    driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
        'source': '''
            Object.defineProperty(navigator, 'webdriver', {
                get: () => undefined
            })
        '''
    })
    
    return driver


 ## 2. 스타벅스 매장찾기 페이지 접속

In [7]:
# 드라이버 생성
driver = get_chrome_driver()

# 스타벅스 매장찾기 페이지 URL
url = "https://www.starbucks.co.kr/store/store_map.do"

# 페이지 접속
driver.get(url)

# 페이지가 완전히 로드될 때까지 잠시 대기
time.sleep(3)

print(f"페이지 접속 완료: {driver.title}")

페이지 접속 완료: Starbucks Korea


 ## 3. 매장 상세 정보 팝업 열기

In [8]:
# 매장 ID 3593번 (대전탄방역점) - HTML 예시와 동일한 매장
store_id = 3593

# JavaScript 함수 실행 (getStoreDetail 함수 호출)
script = f"getStoreDetail('{store_id}')"
driver.execute_script(script)

# 팝업이 로드될 때까지 잠시 대기
time.sleep(2)

print(f"매장 {store_id}번의 상세 정보 팝업이 열렸습니다.")

매장 3593번의 상세 정보 팝업이 열렸습니다.


 ## 4. 매장 정보 팝업 요소 찾기

In [9]:
# WebDriverWait 객체 생성 (최대 5초 대기)
wait = WebDriverWait(driver, 5)

# 매장 정보 팝업의 메인 컨테이너 찾기
store_info = wait.until(
    EC.presence_of_element_located((By.CLASS_NAME, "shopArea_pop01"))
)

print("매장 정보 팝업 요소를 찾았습니다!")

매장 정보 팝업 요소를 찾았습니다!


 ## 5. 기본 정보 추출 (매장명, 주소, 전화번호)

In [10]:
# 1. 매장명 추출
store_name = store_info.find_element(By.CSS_SELECTOR, "header.titl h6").text
print(f"매장명: {store_name}")

# 2. 주소 추출 (첫 번째 dl.shopArea_info)
address_element = store_info.find_element(By.CSS_SELECTOR, "dl.shopArea_info dd")
address = address_element.text
print(f"주소: {address}")

# 3. 전화번호 추출 (두 번째 dl.shopArea_info)
# 'dl_tel' 클래스가 있는 요소를 직접 찾기
tel_element = store_info.find_element(By.CSS_SELECTOR, "dl.shopArea_info.dl_tel dd")
tel = tel_element.text
print(f"전화번호: {tel}")

매장명: 대전탄방역
주소: 대전광역시 서구 문정로2번길 95 (탄방동)
대전광역시 서구 탄방동 673 주안빌딩
전화번호: 1522-3232 (평일, 09:00~18:00)


 ## 6. 영업시간 정보 추출



 **중요 변경 사항:**

 - 영업시간이 탭(tab) 내부에 숨겨져 있음

 - `cafetimeWrap` 클래스 내부의 구조화된 데이터 추출

 - 좌/우 영역으로 나뉘어진 영업시간 정보 통합

In [11]:
try:
    # 영업시간 탭 영역 찾기
    cafe_time_wrap = store_info.find_element(By.CLASS_NAME, "cafetimeWrap")
    
    # 영업시간 정보를 저장할 딕셔너리 (요일을 키로 사용)
    business_hours_dict = {}
    
    # 왼쪽 영역의 영업시간
    try:
        date_time_left = cafe_time_wrap.find_element(By.CLASS_NAME, "date_time_left")
        dt_elements = date_time_left.find_elements(By.TAG_NAME, "dt")
        dd_elements = date_time_left.find_elements(By.TAG_NAME, "dd")
        
        for dt, dd in zip(dt_elements, dd_elements):
            # dd.text에서 요일과 시간 추출
            # 예: "수요일  07:00 ~ 22:00"
            dd_text = dd.text.strip()
            parts = dd_text.split(maxsplit=1)  # 첫 공백을 기준으로 분리
            
            if len(parts) >= 2:
                day_of_week = parts[0]  # 요일 (예: "수요일")
                time_range = parts[1].strip()  # 시간 (예: "07:00 ~ 22:00")
                business_hours_dict[day_of_week] = time_range
    except:
        pass
    
    # 오른쪽 영역의 영업시간
    try:
        date_time_right = cafe_time_wrap.find_element(By.CLASS_NAME, "date_time_right")
        dt_elements = date_time_right.find_elements(By.TAG_NAME, "dt")
        dd_elements = date_time_right.find_elements(By.TAG_NAME, "dd")
        
        for dt, dd in zip(dt_elements, dd_elements):
            dd_text = dd.text.strip()
            parts = dd_text.split(maxsplit=1)
            
            if len(parts) >= 2:
                day_of_week = parts[0]
                time_range = parts[1].strip()
                business_hours_dict[day_of_week] = time_range
    except:
        pass
    
    # 월~일 순서로 재배열
    day_order = ["월요일", "화요일", "수요일", "목요일", "금요일", "토요일", "일요일"]
    business_hours_list = []
    
    for day in day_order:
        if day in business_hours_dict:
            business_hours_list.append(f"{day} {business_hours_dict[day]}")
    
    # 리스트를 줄바꿈으로 연결
    business_hours = "\n".join(business_hours_list) if business_hours_list else "정보 없음"
    
    print(f"영업시간:\n{business_hours}")
    
except Exception as e:
    business_hours = "정보 없음"
    print(f"영업시간 정보 추출 실패: {e}")

영업시간:
월요일 07:00 ~ 22:00
화요일 07:00 ~ 22:00
수요일 07:00 ~ 22:00
목요일 07:00 ~ 22:00
금요일 07:00 ~ 22:00
토요일 07:00 ~ 22:00
일요일 07:00 ~ 22:00


 ## 7. 주차 정보 추출

In [12]:
try:
    # '주차정보' dt를 가진 dl 요소 찾기
    parking_dls = store_info.find_elements(By.CSS_SELECTOR, "dl.shopArea_info")
    parking_info = None
    
    for dl in parking_dls:
        try:
            dt = dl.find_element(By.TAG_NAME, "dt")
            if "주차정보" in dt.text or "주차" in dt.text:
                dd = dl.find_element(By.TAG_NAME, "dd")
                parking_info = dd.text
                break
        except:
            continue
    
    if not parking_info:
        parking_info = "정보 없음"
    
    print(f"주차 정보: {parking_info}")
    
except Exception as e:
    parking_info = "정보 없음"
    print(f"주차 정보 추출 실패: {e}")


주차 정보: 1.주차가능 2.주차장 위치-건물내 주차장 3.주차가능대수-92대 4.주차조건-1시간 무료 5.주차요금정산방법-출차 시 영수증제시


 ## 8. 오시는 길 정보 추출

In [13]:
try:
    # '오시는 길' dt를 가진 dl 요소 찾기
    directions_dls = store_info.find_elements(By.CSS_SELECTOR, "dl.shopArea_info")
    directions = None
    
    for dl in directions_dls:
        try:
            dt = dl.find_element(By.TAG_NAME, "dt")
            if "오시는 길" in dt.text:
                dd = dl.find_element(By.TAG_NAME, "dd")
                directions = dd.text
                break
        except:
            continue
    
    if not directions:
        directions = "정보 없음"
    
    print(f"오시는 길: {directions}")
    
except Exception as e:
    directions = "정보 없음"
    print(f"오시는 길 정보 추출 실패: {e}")

오시는 길: 정보 없음


 ## 9. 수집한 데이터를 딕셔너리로 구성

In [14]:
from pprint import pprint

# 매장 정보를 담을 딕셔너리 생성
store_data = {
    "id": store_id,
    "name": store_name,
    "address": address,
    "tel": tel,
    "business_hours": business_hours,
    "parking_info": parking_info,
    "directions": directions
}

# 결과 출력
print("\n" + "="*80)
print("[수집된 매장 정보]")
print("="*80)
pprint(store_data)


[수집된 매장 정보]
{'address': '대전광역시 서구 문정로2번길 95 (탄방동)\n대전광역시 서구 탄방동 673 주안빌딩',
 'business_hours': '월요일 07:00 ~ 22:00\n'
                   '화요일 07:00 ~ 22:00\n'
                   '수요일 07:00 ~ 22:00\n'
                   '목요일 07:00 ~ 22:00\n'
                   '금요일 07:00 ~ 22:00\n'
                   '토요일 07:00 ~ 22:00\n'
                   '일요일 07:00 ~ 22:00',
 'directions': '정보 없음',
 'id': 3593,
 'name': '대전탄방역',
 'parking_info': '1.주차가능 2.주차장 위치-건물내 주차장 3.주차가능대수-92대 4.주차조건-1시간 무료 '
                 '5.주차요금정산방법-출차 시 영수증제시',
 'tel': '1522-3232 (평일, 09:00~18:00)'}


 ## 12. 팝업 닫기

In [15]:
# 팝업 닫기 버튼 찾기
close_button = store_info.find_element(By.CSS_SELECTOR, "p.btn_pop_close a.isStoreViewClosePop")
close_button.click()
time.sleep(0.5)

print("팝업을 닫았습니다.")

팝업을 닫았습니다.


 ## 13. 여러 매장 정보 반복 수집 (통합 함수)

In [16]:
def extract_store_info(driver, store_id):
    """
    매장 ID를 받아서 해당 매장의 상세 정보를 추출하는 함수
    
    Args:
        driver: Selenium WebDriver 객체
        store_id: 매장 ID (숫자 또는 문자열)
    
    Returns:
        dict: 매장 정보 딕셔너리
    """
    try:
        # 1. JavaScript 함수 실행하여 팝업 열기
        script = f"getStoreDetail('{store_id}')"
        driver.execute_script(script)
        
        # 2. 팝업 로드 대기
        wait = WebDriverWait(driver, 5)
        store_info = wait.until(
            EC.presence_of_element_located((By.CLASS_NAME, "shopArea_pop01"))
        )
        
        # 3. 기본 정보 추출
        store_name = store_info.find_element(By.CSS_SELECTOR, "header.titl h6").text
        address = store_info.find_element(By.CSS_SELECTOR, "dl.shopArea_info dd").text
        
        # 4. 전화번호 추출
        try:
            tel = store_info.find_element(By.CSS_SELECTOR, "dl.shopArea_info.dl_tel dd").text
        except:
            tel = "정보 없음"
        
        # 5. 영업시간 추출
        business_hours_list = []
        try:
            cafe_time_wrap = store_info.find_element(By.CLASS_NAME, "cafetimeWrap")
            
            # 왼쪽 영역
            try:
                date_time_left = cafe_time_wrap.find_element(By.CLASS_NAME, "date_time_left")
                dt_elements = date_time_left.find_elements(By.TAG_NAME, "dt")
                dd_elements = date_time_left.find_elements(By.TAG_NAME, "dd")
                for dt, dd in zip(dt_elements, dd_elements):
                    business_hours_list.append(f"{dt.text.strip()} {dd.text.strip()}")
            except:
                pass
            
            # 오른쪽 영역
            try:
                date_time_right = cafe_time_wrap.find_element(By.CLASS_NAME, "date_time_right")
                dt_elements = date_time_right.find_elements(By.TAG_NAME, "dt")
                dd_elements = date_time_right.find_elements(By.TAG_NAME, "dd")
                for dt, dd in zip(dt_elements, dd_elements):
                    business_hours_list.append(f"{dt.text.strip()} {dd.text.strip()}")
            except:
                pass
            
            business_hours = "\n".join(business_hours_list) if business_hours_list else "정보 없음"
        except:
            business_hours = "정보 없음"
        
        # 6. 주차 정보 추출
        parking_info = "정보 없음"
        try:
            parking_dls = store_info.find_elements(By.CSS_SELECTOR, "dl.shopArea_info")
            for dl in parking_dls:
                try:
                    dt = dl.find_element(By.TAG_NAME, "dt")
                    if "주차" in dt.text:
                        parking_info = dl.find_element(By.TAG_NAME, "dd").text
                        break
                except:
                    continue
        except:
            pass
        
        # 7. 오시는 길 정보 추출
        directions = "정보 없음"
        try:
            directions_dls = store_info.find_elements(By.CSS_SELECTOR, "dl.shopArea_info")
            for dl in directions_dls:
                try:
                    dt = dl.find_element(By.TAG_NAME, "dt")
                    if "오시는 길" in dt.text:
                        directions = dl.find_element(By.TAG_NAME, "dd").text
                        break
                except:
                    continue
        except:
            pass
        
        # 8. 데이터 구성
        store_data = {
            "id": store_id,
            "name": store_name,
            "address": address,
            "tel": tel,
            "business_hours": business_hours,
            "parking_info": parking_info,
            "directions": directions
        }
        
        # 9. 팝업 닫기
        close_button = store_info.find_element(By.CSS_SELECTOR, "p.btn_pop_close a.isStoreViewClosePop")
        close_button.click()
        time.sleep(0.5)
        
        return store_data
        
    except Exception as e:
        print(f"매장 {store_id} 추출 실패: {e}")
        return None

print("매장 정보 추출 함수 정의 완료!")

매장 정보 추출 함수 정의 완료!


 ## 14. 여러 매장 정보 수집 실행

In [17]:
# 수집할 매장 정보 리스트
store_list = []

print(f"\n{'='*80}")

for store_id in range(1, 10):
    print(f"{store_id}번 처리 중...")
    
    store_data = extract_store_info(driver, store_id)
    
    if store_data:
        store_list.append(store_data)
        print(f"성공: {store_data['name']}\n")
    else:
        print(f"실패: 매장 {store_id}번\n")

print(f"{'='*80}")
print(f"수집 완료! 총 {len(store_list)}개 매장 정보 수집됨")
print(f"{'='*80}")


1번 처리 중...
매장 1 추출 실패: Message: 
Stacktrace:
0   chromedriver                        0x00000001012efcf8 cxxbridge1$str$ptr + 2895872
1   chromedriver                        0x00000001012e7c34 cxxbridge1$str$ptr + 2862908
2   chromedriver                        0x0000000100e0d570 _RNvCs47EqcsrPRmA_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 74324
3   chromedriver                        0x0000000100e54f34 _RNvCs47EqcsrPRmA_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 367640
4   chromedriver                        0x0000000100e963d8 _RNvCs47EqcsrPRmA_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 635068
5   chromedriver                        0x0000000100e490f8 _RNvCs47EqcsrPRmA_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 318940
6   chromedriver                        0x00000001012b381c cxxbridge1$str$ptr + 2648868
7   chromedriver                        0x00000001012b6df8 cxxbridge1$str$ptr + 2662656
8   chromedriver                        0x0000000101294334 cxxbridg

 ## 15. 수집한 데이터 확인

In [18]:
from pprint import pprint

print("\n[수집된 전체 매장 정보]")
for idx, store in enumerate(store_list, 1):
    print(f"\n{'='*80}")
    print(f"{idx}. {store['name']}")
    print(f"{'='*80}")
    pprint(store)


[수집된 전체 매장 정보]

1. 학여울역
{'address': '서울특별시 강남구 영동대로 215 (대치동)\n서울특별시 강남구 대치동 994-10',
 'business_hours': '정보 없음',
 'directions': '지하철 3호선 학여울역 1번 출구 대각선 방향',
 'id': 3,
 'name': '학여울역',
 'parking_info': '1. 주차 불가능',
 'tel': '1522-3232 (평일, 09:00~18:00)'}

2. 둔산은하수
{'address': '대전광역시 서구 둔산로31번길 28, 금정빌딩 1층 (둔산동)\n대전광역시 서구 둔산동 1010번지 금정빌딩 1층',
 'business_hours': '정보 없음',
 'directions': '갤러리아백화점 타임월드점 주차동 옆',
 'id': 6,
 'name': '둔산은하수',
 'parking_info': '1.주차불가능',
 'tel': '1522-3232 (평일, 09:00~18:00)'}

3. 신세계강남7층
{'address': '서울특별시 서초구 신반포로 176, 센트럴시티빌딩 7층 (반포동)\n'
            '서울특별시 서초구 반포동 19-3 신세계강남점 7층',
 'business_hours': '정보 없음',
 'directions': '지하철 3,7,9호선 고속터미널역 신세계백화점 신관 7층',
 'id': 7,
 'name': '신세계강남7층',
 'parking_info': '스타벅스 영수증(합산가능) 3만원이상 1시간 무료 주차',
 'tel': '1522-3232 (평일, 09:00~18:00)'}

4. 명동눈스퀘어
{'address': '서울특별시 중구 명동길 14, 눈스퀘어 4층 (명동2가)\n서울특별시 중구 명동2가 83-5 번지 눈스퀘어 4층',
 'business_hours': '정보 없음',
 'directions': '명동 영플라자 건너편 명동 눈스퀘어몰 4층',
 'id': 9,
 'name': '명동눈스퀘어',


 ## 16. JSON 파일로 저장

In [19]:
import pandas as pd

# 리스트를 데이터프레임으로 변환
df = pd.DataFrame(store_list)

# 데이터프레임 정보 출력
print(f"\n{'='*80}")
print(f"\n[데이터 미리보기]")
display(df.head())

# JSON 파일로 저장
filename = "starbucks_stores.json"
df.to_json(filename, orient='records', force_ascii=False, indent=2)

print(f"\n{'='*80}")
print(f"데이터가 '{filename}' 파일로 저장되었습니다.")
print(f"저장된 매장 수: {len(df)}개")
print(f"컬럼: {list(df.columns)}")
print(f"{'='*80}")



[데이터 미리보기]


,id,name,address,tel,business_hours,parking_info,directions
0,3,학여울역,서울특별시 강남구 영동대로 215 (대치동)\n서울특별시 강남구 대치동 994-10,"1522-3232 (평일, 09:00~18:00)",정보 없음,1. 주차 불가능,지하철 3호선 학여울역 1번 출구 대각선 방향
1,6,둔산은하수,"대전광역시 서구 둔산로31번길 28, 금정빌딩 1층 (둔산동)\n대전광역시 서구 둔...","1522-3232 (평일, 09:00~18:00)",정보 없음,1.주차불가능,갤러리아백화점 타임월드점 주차동 옆
2,7,신세계강남7층,"서울특별시 서초구 신반포로 176, 센트럴시티빌딩 7층 (반포동)\n서울특별시 서초...","1522-3232 (평일, 09:00~18:00)",정보 없음,스타벅스 영수증(합산가능) 3만원이상 1시간 무료 주차,"지하철 3,7,9호선 고속터미널역 신세계백화점 신관 7층"
3,9,명동눈스퀘어,"서울특별시 중구 명동길 14, 눈스퀘어 4층 (명동2가)\n서울특별시 중구 명동2가...","1522-3232 (평일, 09:00~18:00)",정보 없음,"건물내 주차장 이용 (10분당 1,000원)",명동 영플라자 건너편 명동 눈스퀘어몰 4층



데이터가 'starbucks_stores.json' 파일로 저장되었습니다.
저장된 매장 수: 4개
컬럼: ['id', 'name', 'address', 'tel', 'business_hours', 'parking_info', 'directions']


 ## 17. 브라우저 종료

In [20]:
driver.quit()
print("브라우저를 종료했습니다.")

브라우저를 종료했습니다.
